## 1) Setup

In [1]:

# If needed, install once:
# %pip install --upgrade sentence-transformers numpy pandas tqdm
# For local LLM via Ollama:
# %pip install --upgrade requests
# Vector store
# %pip install --upgrade chromadb

In [2]:
## langchain imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_core.documents import Document

## vectorstores
from langchain_community.vectorstores import Chroma

## utility imports
import numpy as np
from typing import List

### 1. Sample Data

In [3]:
## create sample documents
sample_docs = [
    """
    Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn
    and improve from experience without being explicitly programmed. There are three main
    types of machine learning: supervised learning, unsupervised learning, and reinforcement
    learning. Supervised learning uses labeled data to train models, while unsupervised
    learning finds patterns in unlabeled data. Reinforcement learning learns through
    interaction with an environment using rewards and penalties.
    """,

    """
    Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks.
    These networks are inspired by the human brain and consist of layers of interconnected
    nodes. Deep learning has revolutionized fields like computer vision, natural language
    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly
    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers
    excel at sequential data processing.
    """,

    """
    Natural Language Processing (NLP)

    NLP is a field of AI that focuses on the interaction between computers and human language.
    Key tasks in NLP include text classification, named entity recognition, sentiment analysis,
    machine translation, and question answering. Modern NLP heavily relies on transformer
    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand
    context and relationships between words in text.
    """
]

sample_docs


['\n    Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn\n    and improve from experience without being explicitly programmed. There are three main\n    types of machine learning: supervised learning, unsupervised learning, and reinforcement\n    learning. Supervised learning uses labeled data to train models, while unsupervised\n    learning finds patterns in unlabeled data. Reinforcement learning learns through\n    interaction with an environment using rewards and penalties.\n    ',
 '\n    Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks.\n    These networks are inspired by the human brain and consist of layers of interconnected\n    nodes. Deep learning has revolutionized fields like computer vision, natural language\n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly\n    effective for image 

In [4]:
## save sample documents to files
import tempfile
temp_dir=tempfile.mkdtemp()

for i,doc in enumerate(sample_docs):
    with open(f"{temp_dir}/doc_{i}.txt","w") as f:
        f.write(doc)

print(f"Sample document created in : {temp_dir}")

Sample document created in : /tmp/tmpjawn8fsc


In [5]:
## save sample documents to files
import tempfile
temp_dir=tempfile.mkdtemp()

for i,doc in enumerate(sample_docs):
    with open(f"doc_{i}.txt","w") as f:
        f.write(doc)

temp_dir

'/tmp/tmp6ov4dwdr'

### 2. Document Loading

In [6]:
from langchain_community.document_loaders import DirectoryLoader,TextLoader

# Load documents from directory
loader = DirectoryLoader(
    "data",
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={'encoding': 'utf-8'}
)
documents = loader.load()

print(f"Loaded {len(documents)} documents")
print(f"\nFirst document preview:")
print(documents[0].page_content[:200] + "...")


Loaded 3 documents

First document preview:

    Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn
    and improve from experience without being explicitly programmed. There...


In [7]:
documents

[Document(metadata={'source': 'data/doc_0.txt'}, page_content='\n    Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn\n    and improve from experience without being explicitly programmed. There are three main\n    types of machine learning: supervised learning, unsupervised learning, and reinforcement\n    learning. Supervised learning uses labeled data to train models, while unsupervised\n    learning finds patterns in unlabeled data. Reinforcement learning learns through\n    interaction with an environment using rewards and penalties.\n    '),
 Document(metadata={'source': 'data/doc_1.txt'}, page_content='\n    Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks.\n    These networks are inspired by the human brain and consist of layers of interconnected\n    nodes. Deep learning has revolutionized fields like computer vision, natural langua

### Document Splitting

In [8]:
# Initialize text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,  # Maximum size of each chunk
    chunk_overlap=50,  # Overlap between chunks to maintain context
    length_function=len,
    separators=[" "]  # Hierarchy of separators
)
chunks=text_splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks from {len(documents)} documents")
print(f"\nChunk example:")
print(f"Content: {chunks[0].page_content[:150]}...")
print(f"Metadata: {chunks[0].metadata}")

Created 5 chunks from 3 documents

Chunk example:
Content: Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn
    and improve from experien...
Metadata: {'source': 'data/doc_0.txt'}


In [9]:
chunks

[Document(metadata={'source': 'data/doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn\n    and improve from experience without being explicitly programmed. There are three main\n    types of machine learning: supervised learning, unsupervised learning, and reinforcement\n    learning. Supervised learning uses labeled data to train models, while unsupervised\n    learning finds patterns in unlabeled data. Reinforcement learning learns through\n    interaction'),
 Document(metadata={'source': 'data/doc_0.txt'}, page_content='learning learns through\n    interaction with an environment using rewards and penalties.'),
 Document(metadata={'source': 'data/doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks.\n    These networks are inspired by the human brain and consist of layers of interconnected

### Embedding Models

In [10]:
sample_text="Machine LEarning is fascinating"

In [11]:
### Huggingface model

from langchain_huggingface import HuggingFaceEmbeddings

## Initialize a simple Embedding model(no API Key needed!)
embeddings=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
embeddings

HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [12]:
vector=embeddings.embed_query(sample_text)
vector

[-0.04763009399175644,
 -0.09523383527994156,
 0.08622037619352341,
 0.008249843493103981,
 0.01219354197382927,
 -0.0504952035844326,
 -0.007320767734199762,
 -0.05885447934269905,
 -0.039019085466861725,
 -0.028660986572504044,
 -0.08570093661546707,
 0.07921122759580612,
 0.004685709252953529,
 -0.02150491438806057,
 -0.07088702917098999,
 0.01464945450425148,
 -0.013478760607540607,
 -0.015381771139800549,
 -0.08215577155351639,
 -0.12226203083992004,
 -0.0034171873703598976,
 0.013044759631156921,
 -0.00855389516800642,
 0.007963662035763264,
 0.02498508431017399,
 0.018824325874447823,
 0.06268896162509918,
 0.019554147496819496,
 0.04684525728225708,
 -0.0665501281619072,
 0.003416009247303009,
 0.06802241504192352,
 -0.011118336580693722,
 0.03004435822367668,
 -0.09648733586072922,
 0.02735748514533043,
 0.009005110710859299,
 0.017103243619203568,
 0.04554048925638199,
 0.006371502764523029,
 -0.005477485712617636,
 -0.04323988035321236,
 0.03637196123600006,
 -0.004953178577

### Intilialize the ChromaDB Vector Store And store the chunks in Vector Representation

In [13]:
chunks

[Document(metadata={'source': 'data/doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn\n    and improve from experience without being explicitly programmed. There are three main\n    types of machine learning: supervised learning, unsupervised learning, and reinforcement\n    learning. Supervised learning uses labeled data to train models, while unsupervised\n    learning finds patterns in unlabeled data. Reinforcement learning learns through\n    interaction'),
 Document(metadata={'source': 'data/doc_0.txt'}, page_content='learning learns through\n    interaction with an environment using rewards and penalties.'),
 Document(metadata={'source': 'data/doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks.\n    These networks are inspired by the human brain and consist of layers of interconnected

In [14]:
## Create a Chromdb vector store
persist_directory="./chroma_db"

## Initialize Chromadb with HuggingFace embeddings
vectorstore=Chroma.from_documents(
    documents=chunks,
    embedding=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"),
    persist_directory=persist_directory,
    collection_name="rag_collection"

)

print(f"Vector store created with {vectorstore._collection.count()} vectors")
print(f"Vector store name: {vectorstore._collection.name}")
print(f"Persisted to: {persist_directory}")

Vector store created with 102 vectors
Vector store name: rag_collection
Persisted to: ./chroma_db


### Test Similarity Search

In [15]:
query="What are the types of machine learning?"

similar_docs=vectorstore.similarity_search(query,k=3)
similar_docs

[Document(metadata={'source': 'data/doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn\n    and improve from experience without being explicitly programmed. There are three main\n    types of machine learning: supervised learning, unsupervised learning, and reinforcement\n    learning. Supervised learning uses labeled data to train models, while unsupervised\n    learning finds patterns in unlabeled data. Reinforcement learning learns through\n    interaction'),
 Document(metadata={'source': 'data/doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn\n    and improve from experience without being explicitly programmed. There are three main\n    types of machine learning: supervised learning, unsupervised learning, and reinforcement\n    learning. Supervised learning uses labeled data to train m

In [16]:
query="what is NLP?"

similar_docs=vectorstore.similarity_search(query,k=3)
similar_docs

[Document(metadata={'source': 'data/doc_2.txt'}, page_content='Natural Language Processing (NLP)\n\n    NLP is a field of AI that focuses on the interaction between computers and human language.\n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis,\n    machine translation, and question answering. Modern NLP heavily relies on transformer\n    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand\n    context and relationships between words in text.'),
 Document(metadata={'source': 'data/doc_2.txt'}, page_content='Natural Language Processing (NLP)\n\n    NLP is a field of AI that focuses on the interaction between computers and human language.\n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis,\n    machine translation, and question answering. Modern NLP heavily relies on transformer\n    architectures like BERT, GPT, and T5. These models use attention mechanisms t

In [17]:
query="what is Deep Learning?"

similar_docs=vectorstore.similarity_search(query,k=3)
similar_docs

[Document(metadata={'source': 'data/doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks.\n    These networks are inspired by the human brain and consist of layers of interconnected\n    nodes. Deep learning has revolutionized fields like computer vision, natural language\n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly\n    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers'),
 Document(metadata={'source': 'data/doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks.\n    These networks are inspired by the human brain and consist of layers of interconnected\n    nodes. Deep learning has revolutionized fields like computer vision, natural language\n    processing, and speech recognition. Convolutional Neural Network

In [18]:
print(f"Query: {query}")
print(f"\nTop {len(similar_docs)} similar chunks:")
for i, doc in enumerate(similar_docs):
    print(f"\n--- Chunk {i+1} ---")
    print(doc.page_content[:200] + "...")
    print(f"Source: {doc.metadata.get('source', 'Unknown')}")

Query: what is Deep Learning?

Top 3 similar chunks:

--- Chunk 1 ---
Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks.
    These networks are inspired by the human brain and consist of layers of in...
Source: data/doc_1.txt

--- Chunk 2 ---
Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks.
    These networks are inspired by the human brain and consist of layers of in...
Source: data/doc_1.txt

--- Chunk 3 ---
Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks.
    These networks are inspired by the human brain and consist of layers of in...
Source: data/doc_1.txt


### Advanced Similarity Search With Scores

In [19]:
results_scores=vectorstore.similarity_search_with_score(query,k=3)
results_scores

[(Document(metadata={'source': 'data/doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks.\n    These networks are inspired by the human brain and consist of layers of interconnected\n    nodes. Deep learning has revolutionized fields like computer vision, natural language\n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly\n    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers'),
  0.580187976360321),
 (Document(metadata={'source': 'data/doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks.\n    These networks are inspired by the human brain and consist of layers of interconnected\n    nodes. Deep learning has revolutionized fields like computer vision, natural language\n    processing, and speech recognition. Conv

#### Understanding Similarity Scores
The similarity score represents how closely related a document chunk is to your query. The scoring depends on the distance metric used:

ChromaDB default: Uses L2 distance (Euclidean distance)

- Lower scores = MORE similar (closer in vector space)
- Score of 0 = identical vectors
- Typical range: 0 to 2 (but can be higher)


Cosine similarity (if configured):

- Higher scores = MORE similar
- Range: -1 to 1 (1 being identical)

#### Initialize LLM, RAG Chain, Prompt Template,Query the RAG system

In [20]:
from langchain_community.llms import Ollama

#OLLAMA_URL = os.getenv("OLLAMA_URL", "http://localhost:11434")
#OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3.1:8b")

llm = Ollama(model="llama3.1:8b")

/tmp/ipykernel_3493754/3842857748.py:6: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model="llama3.1:8b")


In [21]:
# Use the LLM to generate prompt
test_response=llm.invoke("What are Large Language Models")
test_response

"Large Language Models (LLMs) are a type of artificial intelligence (AI) designed to process and generate human-like language. They are a subset of Natural Language Processing (NLP) models, which are trained on vast amounts of text data to understand the structure and semantics of language.\n\n**Key Characteristics:**\n\n1. **Scale**: LLMs are typically massive neural networks with hundreds of millions or even billions of parameters.\n2. **Depth**: They have multiple layers, often with multiple attention mechanisms, enabling them to capture complex relationships between words and concepts.\n3. **Training data**: LLMs are trained on enormous datasets, including books, articles, websites, and user-generated content.\n4. **Language understanding**: They can comprehend language at various levels, from syntax and semantics to pragmatics and discourse.\n\n**How they work:**\n\n1. **Input**: Users provide input in the form of text, which is then processed by the LLM's neural network.\n2. **To

In [22]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.1:8b",
    temperature=0
)
llm

ChatOllama(model='llama3.1:8b', temperature=0.0)

In [23]:
llm.invoke("What is AI")

AIMessage(content='Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think and learn like humans. The term can also be applied to any machine that exhibits traits associated with a human mind such as learning and problem-solving.\n\nAI technology is based on the principle of creating algorithms that enable computers to perform tasks that would typically require human intelligence, including:\n\n1. **Learning**: AI systems can learn from data, experiences, or interactions.\n2. **Problem-solving**: AI systems can identify problems and develop solutions.\n3. **Reasoning**: AI systems can draw conclusions based on available information.\n4. **Perception**: AI systems can interpret sensory data (e.g., images, speech).\n\nAI has many applications across various industries, including:\n\n1. **Virtual assistants** (e.g., Siri, Alexa)\n2. **Image recognition** (e.g., facial recognition, object detection)\n3. **Natural language process

### Create RAG Chain Alternative - Using LCEL (LangChain Expression Language)

In [24]:
## Convert vector store to retriever
retriever=vectorstore.as_retriever(
     search_kwarg={"k":3} ## Retrieve top 3 relevant chunks
 )
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x1534c388f3b0>, search_kwargs={})

In [25]:
# Even more flexible approach using LCEL
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

In [26]:
from langchain_core.prompts import ChatPromptTemplate

# Create a custom prompt
custom_prompt = ChatPromptTemplate.from_template("""You are a concise, technical assistant for Amber molecular simulation users. "
    Answer the user's question using ONLY the provided context.
    If the answer cannot be determined from the context, say you do not know.
    Cite sources using [Title#chunkN] notation.
    Do not speculate or introduce external knowledge.

Context:
{context}

Question: {question}

Answer:""")
custom_prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='You are a concise, technical assistant for Amber molecular simulation users. "\n    Answer the user\'s question using ONLY the provided context.\n    If the answer cannot be determined from the context, say you do not know.\n    Cite sources using [Title#chunkN] notation.\n    Do not speculate or introduce external knowledge.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:'), additional_kwargs={})])

In [27]:
## Convert vector store to retriever
retriever=vectorstore.as_retriever(
     search_kwarg={"k":3} ## Retrieve top 3 relevant chunks
 )

retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x1534c388f3b0>, search_kwargs={})

In [28]:
## Format the output documents for the prompt
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [29]:
## Build the chain ussing LCEL

rag_chain_lcel=(
    {
        "context":retriever | format_docs,
        "question": RunnablePassthrough()
     }
    | custom_prompt
    | llm
    | StrOutputParser()
)

rag_chain_lcel

{
  context: VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x1534c388f3b0>, search_kwargs={})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='You are a concise, technical assistant for Amber molecular simulation users. "\n    Answer the user\'s question using ONLY the provided context.\n    If the answer cannot be determined from the context, say you do not know.\n    Cite sources using [Title#chunkN] notation.\n    Do not speculate or introduce external knowledge.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:'), additional_kwargs={})])
| ChatOllama(model='llama3.1:8b', temperature=0.0)
| StrOutputParser()

In [30]:
response=rag_chain_lcel.invoke("What is Deep Learning")
response

'Deep learning is a subset of machine learning based on artificial neural networks [Source: Context].'

In [31]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
docs = retriever.invoke("What is Deep Learning")


In [32]:
#retriever.get_relevant_documents("What is Deep Learning")
retDocs = retriever.invoke("What is Deep Learning")
retDocs

[Document(metadata={'source': 'data/doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks.\n    These networks are inspired by the human brain and consist of layers of interconnected\n    nodes. Deep learning has revolutionized fields like computer vision, natural language\n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly\n    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers'),
 Document(metadata={'source': 'data/doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks.\n    These networks are inspired by the human brain and consist of layers of interconnected\n    nodes. Deep learning has revolutionized fields like computer vision, natural language\n    processing, and speech recognition. Convolutional Neural Network

In [33]:
# Query using the LCEL approach - Fixed version
def query_rag_lcel(question):
    print(f"Question: {question}")
    print("-" * 50)

    # Method 1: Pass string directly (when using RunnablePassthrough)
    answer = rag_chain_lcel.invoke(question)
    print(f"Answer: {answer}")

    # Get source documents separately if needed
    docs = retriever.invoke(question)
    #docs = retriever.get_relevant_documents(question)
    print("\nSource Documents:")
    for i, doc in enumerate(docs):
        print(f"\n--- Source {i+1} ---")
        print(doc.page_content[:200] + "...")

In [34]:
# Test LCEL chain
print("Testing LCEL Chain:")
query_rag_lcel("What are the key concepts in reinforcement learning?")

Testing LCEL Chain:
Question: What are the key concepts in reinforcement learning?
--------------------------------------------------
Answer: The key concepts in reinforcement learning include: states, actions, rewards, policies, and value functions. [Reinforcement Learning in Detail#chunk1]

Source Documents:

--- Source 1 ---
Reinforcement Learning in Detail

Reinforcement learning (RL) is a type of machine learning where an agent learns to make
decisions by interacting with an environment. The agent receives rewards or pe...

--- Source 2 ---
Reinforcement Learning in Detail

Reinforcement learning (RL) is a type of machine learning where an agent learns to make
decisions by interacting with an environment. The agent receives rewards or pe...

--- Source 3 ---
Reinforcement Learning in Detail

Reinforcement learning (RL) is a type of machine learning where an agent learns to make
decisions by interacting with an environment. The agent receives rewards or pe...


In [35]:
query_rag_lcel("What is machine learning?")

Question: What is machine learning?
--------------------------------------------------
Answer: Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed [Source 1#chunk1]. 

[Source 1: Machine Learning Fundamentals]

Source Documents:

--- Source 1 ---
Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn
    and improve from experience without being explicitly programmed. There are ...

--- Source 2 ---
Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn
    and improve from experience without being explicitly programmed. There are ...

--- Source 3 ---
Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn
    and improve from experience without being explicitly programmed. There are ...


### Add New Documents To Existing Vector Store

In [36]:
vectorstore

In [37]:
# Add new documents to the existing vector store
new_document = """
Reinforcement Learning in Detail

Reinforcement learning (RL) is a type of machine learning where an agent learns to make
decisions by interacting with an environment. The agent receives rewards or penalties
based on its actions and learns to maximize cumulative reward over time. Key concepts
in RL include: states, actions, rewards, policies, and value functions. Popular RL
algorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and
Actor-Critic methods. RL has been successfully applied to game playing (like AlphaGo),
robotics, and autonomous systems.
"""

In [38]:
new_document

'\nReinforcement Learning in Detail\n\nReinforcement learning (RL) is a type of machine learning where an agent learns to make\ndecisions by interacting with an environment. The agent receives rewards or penalties\nbased on its actions and learns to maximize cumulative reward over time. Key concepts\nin RL include: states, actions, rewards, policies, and value functions. Popular RL\nalgorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and\nActor-Critic methods. RL has been successfully applied to game playing (like AlphaGo),\nrobotics, and autonomous systems.\n'

In [39]:
chunks

[Document(metadata={'source': 'data/doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn\n    and improve from experience without being explicitly programmed. There are three main\n    types of machine learning: supervised learning, unsupervised learning, and reinforcement\n    learning. Supervised learning uses labeled data to train models, while unsupervised\n    learning finds patterns in unlabeled data. Reinforcement learning learns through\n    interaction'),
 Document(metadata={'source': 'data/doc_0.txt'}, page_content='learning learns through\n    interaction with an environment using rewards and penalties.'),
 Document(metadata={'source': 'data/doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks.\n    These networks are inspired by the human brain and consist of layers of interconnected

In [40]:
new_doc=Document(
    page_content=new_document,
    metadata={"source": "manual_addition", "topic": "reinforcement_learning"}
)

In [41]:
new_doc

Document(metadata={'source': 'manual_addition', 'topic': 'reinforcement_learning'}, page_content='\nReinforcement Learning in Detail\n\nReinforcement learning (RL) is a type of machine learning where an agent learns to make\ndecisions by interacting with an environment. The agent receives rewards or penalties\nbased on its actions and learns to maximize cumulative reward over time. Key concepts\nin RL include: states, actions, rewards, policies, and value functions. Popular RL\nalgorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and\nActor-Critic methods. RL has been successfully applied to game playing (like AlphaGo),\nrobotics, and autonomous systems.\n')

In [42]:
## split the documents
new_chunks=text_splitter.split_documents([new_doc])
new_chunks

[Document(metadata={'source': 'manual_addition', 'topic': 'reinforcement_learning'}, page_content='Reinforcement Learning in Detail\n\nReinforcement learning (RL) is a type of machine learning where an agent learns to make\ndecisions by interacting with an environment. The agent receives rewards or penalties\nbased on its actions and learns to maximize cumulative reward over time. Key concepts\nin RL include: states, actions, rewards, policies, and value functions. Popular RL\nalgorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and\nActor-Critic methods. RL has been'),
 Document(metadata={'source': 'manual_addition', 'topic': 'reinforcement_learning'}, page_content='methods, and\nActor-Critic methods. RL has been successfully applied to game playing (like AlphaGo),\nrobotics, and autonomous systems.')]

In [43]:
# Number of vectors in Vectore Store collection
print(f"Total vectors now: {vectorstore._collection.count()}")

Total vectors now: 102


In [44]:
### Add new documents to vectorstore
vectorstore.add_documents(new_chunks)



['bf730096-05a5-4e38-96bf-df9af25da1b5',
 '20e1c357-46a1-4c0d-920c-fae0597eb39c']

In [45]:
print(f"Added {len(new_chunks)} new chunks to the vector store")
print(f"Total vectors now: {vectorstore._collection.count()}")

Added 2 new chunks to the vector store
Total vectors now: 104


In [46]:
## query with the updated vector
new_question="What are the keys concepts in reinforcement learning"
result=query_rag_lcel(new_question)
result

Question: What are the keys concepts in reinforcement learning
--------------------------------------------------
Answer: The key concepts in Reinforcement Learning (RL) include:

* States
* Actions
* Rewards
* Policies
* Value functions [Reinforcement Learning in Detail#chunk1]

Source Documents:

--- Source 1 ---
Reinforcement Learning in Detail

Reinforcement learning (RL) is a type of machine learning where an agent learns to make
decisions by interacting with an environment. The agent receives rewards or pe...

--- Source 2 ---
Reinforcement Learning in Detail

Reinforcement learning (RL) is a type of machine learning where an agent learns to make
decisions by interacting with an environment. The agent receives rewards or pe...

--- Source 3 ---
Reinforcement Learning in Detail

Reinforcement learning (RL) is a type of machine learning where an agent learns to make
decisions by interacting with an environment. The agent receives rewards or pe...


### Advanced Rag Techniques- Conversational Memory
Understanding Conversational Memory in RAG
Conversational memory enables RAG systems to maintain context across multiple interactions. This is crucial for:

Follow-up questions that reference previous answers
Pronoun resolution (e.g., "it", "they", "that")
Context-dependent queries that build on prior discussion
Natural dialogue flow where users don't repeat context

Key Challenge:
Traditional RAG retrieves documents based only on the current query, missing important context from the conversation. For example:

User: "Tell me about Python"
Bot: explains Python programming language
User: "What are its main libraries?" ← "its" refers to Python, but retriever doesn't know this

Solution:
The modern approach uses a two-step process:

Query Reformulation: Transform context-dependent questions into standalone queries
Context-Aware Retrieval: Use the reformulated query to fetch relevant documents

- create_history_aware_retriever: Makes the retriever understand conversation context
- MessagesPlaceholder: Placeholder for chat history in prompts
- HumanMessage/AIMessage: Structured message types for conversation history

In [47]:
from langchain_classic.chains.history_aware_retriever import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

In [48]:
## create a prompt that includes the chat history
contextualize_q_system_prompt = """Given a chat history and the latest user question
which might reference context in the chat history, formulate a standalone question
which can be understood without the chat history. Do NOT answer the question,
just reformulate it if needed and otherwise return it as is."""

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

In [49]:
## create history aware retriever
history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_q_prompt
)
history_aware_retriever

RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
| VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x1534c388f3b0>, search_kwargs={'k': 3}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(t

In [50]:
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# Create a new document chain with history
qa_system_prompt = """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use three sentences maximum and keep the answer concise.

Context: {context}"""

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)

# Create conversational RAG chain
conversational_rag_chain = create_retrieval_chain(
    history_aware_retriever,
    question_answer_chain
)
print("Conversational RAG chain created!")

Conversational RAG chain created!


In [51]:
chat_history=[]
# First question
result1 = conversational_rag_chain.invoke({
    "chat_history": chat_history,
    "input": "What is machine learning?"
})
print(f"Q: What is machine learning?")
print(f"A: {result1['answer']}")

Q: What is machine learning?
A: Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. It allows systems to learn from data and make predictions or decisions based on that information. This process enables the system to adapt and improve over time.


In [52]:
chat_history.extend([
    HumanMessage(content="What is machine learning"),
    AIMessage(content=result1['answer'])
])

In [53]:
chat_history

[HumanMessage(content='What is machine learning', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. It allows systems to learn from data and make predictions or decisions based on that information. This process enables the system to adapt and improve over time.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [54]:
## Follow up question
# Follow-up question
result2 = conversational_rag_chain.invoke({
    "chat_history": chat_history,
    "input": "What are its main types?"  # Refers to ML from previous question
})
result2

{'chat_history': [HumanMessage(content='What is machine learning', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. It allows systems to learn from data and make predictions or decisions based on that information. This process enables the system to adapt and improve over time.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])],
 'input': 'What are its main types?',
 'context': [Document(metadata={'source': 'data/doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn\n    and improve from experience without being explicitly programmed. There are three main\n    types of machine learning: supervised learning, unsupervised learning, and reinforcement\n    learning. Supervised learning uses labeled

In [55]:
result2['answer']

'There are three main types of machine learning: supervised learning, unsupervised learning, and reinforcement learning. Supervised learning uses labeled data to train models, while unsupervised learning finds patterns in unlabeled data. Reinforcement learning learns through interaction with the environment.'